# 2016 BCHH infrasound relative-calibration check

Purpose: test whether the relative gains of the colocated BCHH infrasound channels were stable before and after the 1 September 2016 AMOS-6 explosion.

This notebook deliberately starts from the SQLite event catalog and reads waveform data directly from the SDS archive. It does **not** assume that absolute pressure calibration is correct. The primary measurements are pairwise relative gains, robust amplitude ratios, and correlation.

The curated catalog now distinguishes `event_time_utc` from `window_start/window_end`; AMOS-6 is represented as an `explosion` at 2016-09-01 13:07:11.913 UTC.

In [10]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run notebook from repository root, or change ROOT explicitly.
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DB = ROOT / "data/processed/ksc_rockets.sqlite"

# Set this to the actual SDS root on your machine.
SDS_ROOT = Path("/Volumes/classdata/remastered/SDS_KSC")

XML = Path("/Volumes/classdata/KSC/station_metadata/KSC.xml")

# FLOVOpy import path may differ between environments.
from flovopy.enhanced.sdsclient import EnhancedSDSClient

print("DB:", DB)
print("SDS:", SDS_ROOT)

DB: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/KSCRocketSeismology_steps1-3/data/processed/ksc_rockets.sqlite
SDS: /Volumes/classdata/remastered/SDS_KSC


## 1. Select events around AMOS-6

Start with the launches immediately before and after the explosion. The date range can be widened later to 3–5 usable events on either side.

In [5]:
with sqlite3.connect(DB) as con:
    events = pd.read_sql_query(
        """
        SELECT event_id, event_type, event_time_utc, window_start, window_end,
               mission, vehicle, pad, notes
        FROM events
        WHERE event_time_utc BETWEEN '2016-08-01T00:00:00Z' AND '2016-09-15T23:59:59Z'
        ORDER BY event_time_utc
        """, con
    )
events

,event_id,event_type,event_time_utc,window_start,window_end,mission,vehicle,pad,notes
0,ksc-2e795d04c05e,launch,2016-08-14T05:26:00Z,2016-08-14T05:26:00Z,2016-08-14T07:26:00Z,Falcon 9 Full Thrust | JCSAT-16,Falcon 9,SLC-40,NaN
1,ksc-b391787ae525,launch,2016-08-19T04:47:00Z,2016-08-19T04:47:00Z,2016-08-19T05:52:00Z,"Delta IV M+(4,2) | AFSPC-6",Delta IV,SLC-41,NaN
2,ksc-48988fddd1fa,explosion,2016-09-01T13:07:11.913000Z,2016-09-01T13:00:00Z,2016-09-01T14:00:00Z,Falcon 9 Full Thrust | Amos 6 (Failure before ...,Falcon 9,SLC-40,Corrects erroneous 2016-09-03 source row. Wind...
3,ksc-ba59e6ac6c4f,launch,2016-09-08T23:05:00Z,2016-09-08T23:05:00Z,2016-09-09T01:05:00Z,Atlas V 411 | OSIRIS-REx,Atlas V,SLC-41,NaN


The expected core comparison is JCSAT-16 (14 Aug), AFSPC-6 (19 Aug), AMOS-6 (1 Sep), and OSIRIS-REx (8 Sep). We will let SDS availability decide which events/channels are actually usable.

In [6]:
# Analysis windows relative to event time. These are intentionally independent of
# the catalog's broad extraction windows and can be tuned after waveform inspection.
PRE_S = 30
POST_S = 240

# BCHH pressure channels were encoded differently in different epochs.
# Start broad and inspect actual trace IDs returned by SDS.
NETWORK = "*"
STATION = "BCHH*"
LOCATION = "*"
CHANNEL = "*D*"   # adjust after inspecting availability

sds = EnhancedSDSClient(str(SDS_ROOT))

In [7]:
def read_event(row):
    from obspy import UTCDateTime
    t = UTCDateTime(row.event_time_utc)
    st = sds.read(
        starttime=t-PRE_S, endtime=t+POST_S,
        net=NETWORK, sta=STATION, loc=LOCATION, chan=CHANNEL,
        skip_low_rate_channels=True, merge=None, trim=True,
        postprocess=False, final_smart_merge=False, verbose=False,
    )
    return st

# Inspect IDs first; do not assume DD1/DD2/DD3 until the archive confirms them.
for _, row in events.iterrows():
    try:
        st = read_event(row)
        print(row.event_time_utc, row.mission)
        print("  ", [tr.id for tr in st])
    except Exception as exc:
        print(row.event_time_utc, "READ FAILED:", exc)

2016-08-14T05:26:00Z Falcon 9 Full Thrust | JCSAT-16
   ['1R.BCHH.10.DD1', '1R.BCHH.10.DD2', '1R.BCHH.10.DD3', '1R.BCHH.10.DHE', '1R.BCHH.10.DHN', '1R.BCHH.10.DHZ']
2016-08-19T04:47:00Z Delta IV M+(4,2) | AFSPC-6
   ['1R.BCHH.10.DD1', '1R.BCHH.10.DD2', '1R.BCHH.10.DD3', '1R.BCHH.10.DHE', '1R.BCHH.10.DHN', '1R.BCHH.10.DHZ']
2016-09-01T13:07:11.913000Z Falcon 9 Full Thrust | Amos 6 (Failure before launch)
   ['1R.BCHH.10.DD1', '1R.BCHH.10.DD2', '1R.BCHH.10.DD3', '1R.BCHH.10.DHE', '1R.BCHH.10.DHN', '1R.BCHH.10.DHZ']
2016-09-08T23:05:00Z Atlas V 411 | OSIRIS-REx
   ['1R.BCHH.10.DD1', '1R.BCHH.10.DD2', '1R.BCHH.10.DD3', '1R.BCHH.10.DHE', '1R.BCHH.10.DHN', '1R.BCHH.10.DHZ']


## 2. Response correction

For the relative-calibration test, two complementary calculations are useful:

1. **raw-count relative gain**, which asks whether the acquisition chains changed;
2. **response-corrected relative pressure**, which asks whether the current StationXML makes the channels agree in Pa.

Insert the same StationXML/preprocessing loader used by the ensemble metrics notebook below. Keeping both versions is useful: a change in raw gain but not corrected pressure means the metadata correction is doing its job.

In [11]:
from obspy import read_inventory
inv = read_inventory(XML)

In [12]:
def common_arrays(tr1, tr2):
    """Return finite, time-aligned arrays on the common interval."""
    a=tr1.copy(); b=tr2.copy()
    t0=max(a.stats.starttime,b.stats.starttime); t1=min(a.stats.endtime,b.stats.endtime)
    a.trim(t0,t1,pad=False); b.trim(t0,t1,pad=False)
    # For colocated channels with the same sample rate this should normally suffice.
    n=min(a.stats.npts,b.stats.npts)
    x=np.asarray(a.data[:n],dtype=float); y=np.asarray(b.data[:n],dtype=float)
    good=np.isfinite(x)&np.isfinite(y)
    return x[good],y[good]

def pair_metrics(tr1, tr2):
    x,y=common_arrays(tr1,tr2)
    if len(x)<10: return {}
    # remove offsets; robust through-origin slope after centering
    x=x-np.median(x); y=y-np.median(y)
    denom=np.dot(y,y)
    slope=np.dot(y,x)/denom if denom>0 else np.nan  # tr1 ~= slope * tr2
    rms=lambda z: np.sqrt(np.mean(z*z))
    p99=lambda z: np.percentile(np.abs(z),99)
    return {
        "slope": slope,
        "rms_ratio": rms(x)/rms(y) if rms(y)>0 else np.nan,
        "p99_ratio": p99(x)/p99(y) if p99(y)>0 else np.nan,
        "peak_ratio": np.max(np.abs(x))/np.max(np.abs(y)) if np.max(np.abs(y))>0 else np.nan,
        "correlation": np.corrcoef(x,y)[0,1],
        "npts": len(x),
    }

## 3. Compute pairwise gains

After the trace-ID inspection above, set `PRESSURE_IDS` to the three actual BCHH infrasound channels for this epoch. If sensor serial numbers changed across these dates, retain that information explicitly rather than assuming channel number = sensor identity.

In [13]:
PRESSURE_IDS = [
    "1R.BCHH.10.DD1", "1R.BCHH.10.DD2", "1R.BCHH.10.DD3"
]

def select_ids(st, ids):
    return {tr.id: tr for tr in st if tr.id in ids}

rows=[]
for _, event in events.iterrows():
    st=read_event(event)
    traces=select_ids(st,PRESSURE_IDS)
    ids=sorted(traces)
    for i in range(len(ids)):
        for j in range(i+1,len(ids)):
            m=pair_metrics(traces[ids[i]],traces[ids[j]])
            rows.append({
                "event_id":event.event_id, "event_time_utc":event.event_time_utc,
                "mission":event.mission, "pair":f"{ids[i]} / {ids[j]}", **m
            })
relative=pd.DataFrame(rows)
relative

,event_id,event_time_utc,mission,pair,slope,rms_ratio,p99_ratio,peak_ratio,correlation,npts
0,ksc-2e795d04c05e,2016-08-14T05:26:00Z,Falcon 9 Full Thrust | JCSAT-16,1R.BCHH.10.DD1 / 1R.BCHH.10.DD2,2.518113,40.653434,26.213483,10.915194,0.063749,67501
1,ksc-2e795d04c05e,2016-08-14T05:26:00Z,Falcon 9 Full Thrust | JCSAT-16,1R.BCHH.10.DD1 / 1R.BCHH.10.DD3,-0.027011,2.860734,1.771450,0.773410,-0.009418,67501
2,ksc-2e795d04c05e,2016-08-14T05:26:00Z,Falcon 9 Full Thrust | JCSAT-16,1R.BCHH.10.DD2 / 1R.BCHH.10.DD3,0.008317,0.070369,0.067578,0.070856,0.119130,67501
3,ksc-b391787ae525,2016-08-19T04:47:00Z,"Delta IV M+(4,2) | AFSPC-6",1R.BCHH.10.DD1 / 1R.BCHH.10.DD2,27.271961,42.144031,59.285714,63.339623,0.677285,67501
4,ksc-b391787ae525,2016-08-19T04:47:00Z,"Delta IV M+(4,2) | AFSPC-6",1R.BCHH.10.DD1 / 1R.BCHH.10.DD3,10.403839,93.858264,103.750000,76.295455,0.133328,67501
5,ksc-b391787ae525,2016-08-19T04:47:00Z,"Delta IV M+(4,2) | AFSPC-6",1R.BCHH.10.DD2 / 1R.BCHH.10.DD3,1.213613,2.227083,1.750000,1.204545,0.541786,67501
6,ksc-48988fddd1fa,2016-09-01T13:07:11.913000Z,Falcon 9 Full Thrust | Amos 6 (Failure before ...,1R.BCHH.10.DD1 / 1R.BCHH.10.DD2,17.396044,49.362434,77.666667,10.176417,0.365370,67501
7,ksc-48988fddd1fa,2016-09-01T13:07:11.913000Z,Falcon 9 Full Thrust | Amos 6 (Failure before ...,1R.BCHH.10.DD1 / 1R.BCHH.10.DD3,-0.021282,3.063680,3.174387,0.764839,-0.023146,67501
8,ksc-48988fddd1fa,2016-09-01T13:07:11.913000Z,Falcon 9 Full Thrust | Amos 6 (Failure before ...,1R.BCHH.10.DD2 / 1R.BCHH.10.DD3,0.026826,0.062065,0.040872,0.075158,0.462992,67501
9,ksc-ba59e6ac6c4f,2016-09-08T23:05:00Z,Atlas V 411 | OSIRIS-REx,1R.BCHH.10.DD1 / 1R.BCHH.10.DD2,0.686675,88.221054,42.707692,19.108374,0.006058,67501


In [14]:
if not relative.empty:
    relative["event_time_utc"] = pd.to_datetime(relative.event_time_utc, utc=True)
    for metric in ["slope","rms_ratio","p99_ratio","peak_ratio","correlation"]:
        fig,ax=plt.subplots(figsize=(9,4))
        for pair,g in relative.groupby("pair"):
            ax.plot(g.event_time_utc,g[metric],marker="o",label=pair)
        ax.axvline(pd.Timestamp("2016-09-01T13:07:11.913Z"),ls="--",label="AMOS-6")
        ax.set_ylabel(metric); ax.set_xlabel("UTC"); ax.legend(); fig.autofmt_xdate(); plt.show()

ValueError: time data "2016-09-01T13:07:11.913000Z" doesn't match format "%Y-%m-%dT%H:%M:%S%z". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

## 4. Interpretation checklist

- Do pairwise slopes/ratios remain constant across 1 September?
- Does any step follow a **sensor serial number** or a channel/acquisition path?
- Are ratios stable for ordinary launches but distorted during AMOS-6 because of clipping/non-linearity?
- Do raw-count ratios and response-corrected Pa ratios tell the same story?
- Repeat using moderate-amplitude portions of AMOS-6 if the strongest impulses approach clipping.
- Once stable, write the resulting relative-calibration/QC conclusion back to the station/channel epoch metadata rather than applying an undocumented correction in this notebook.